In [1]:

import os, gc, glob, warnings
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
warnings.filterwarnings('ignore')

SAVE_DIR    = '/srv/work'
IN_LABEL    = f'{SAVE_DIR}/te_labeled_ventanas_v4.parquet'
IN_BASE     = f'{SAVE_DIR}/te_baseline_stats_v4.parquet'
ARCHIVOS    = sorted(glob.glob(f'{SAVE_DIR}/te_*_raw_*.parquet'))
OUT_FEAT    = f'{SAVE_DIR}/te_features_v6.parquet'

METRICS_NET = ['network.latency', 'network.loss', 'network.jitter']
METRICS_BGP = ['bgp.path_changes.count', 'bgp.updates.count', 'bgp.reachability']

METRIC_SHORT = {
    'network.latency': 'lat',
    'network.loss':    'loss',
    'network.jitter':  'jitter',
}
BGP_METRIC_SHORT = {
    'bgp.path_changes.count': 'bgp_path_changes',
    'bgp.updates.count':      'bgp_updates',
    'bgp.reachability':       'bgp_reachability',
}

BD_END  = pd.Timestamp('2026-04-30', tz='UTC')
EPS     = 1e-9
WINDOW  = pd.Timedelta('30min')
H       = 60
W_v6    = 90

ROUTE_KEYS = ['source_agent', 'cloud_provider', 'region']
INDEX_COLS = ROUTE_KEYS + ['window']
TARGET_COL = 'label_ventana'
SPLIT_COL  = 'split'

print(f'Archivos raw : {len(ARCHIVOS)}  (esperado: 72)')
print(f'Labels v4    : {IN_LABEL}')
print(f'Baseline v4  : {IN_BASE}')
print(f'Output       : {OUT_FEAT}')
print(f'BD_END       : {BD_END}')
print(f'W_v6={W_v6}min >= H={H}min  [correcto — lags t-1/t-2/t-48]')

import sys
sys.path.insert(0, '/srv/python_packages')
import mlflow

try:
    mlflow.set_tracking_uri('http://localhost:6000')
    mlflow.set_experiment('latam2cloud')
    MLFLOW_OK = True
    print('MLflow OK — http://localhost:6000  |  experimento: latam2cloud')
except Exception as _e:
    MLFLOW_OK = False
    print(f'MLflow no disponible ({_e}) — continúa sin tracking')

Archivos raw : 75  (esperado: 72)
Labels v4    : /srv/work/te_labeled_ventanas_v4.parquet
Baseline v4  : /srv/work/te_baseline_stats_v4.parquet
Output       : /srv/work/te_features_v6.parquet
BD_END       : 2026-04-30 00:00:00+00:00
W_v6=90min >= H=60min  [correcto — lags t-1/t-2/t-48]
MLflow OK — http://localhost:6000  |  experimento: latam2cloud


In [2]:

labeled_df = pd.read_parquet(IN_LABEL)
labeled_df['window'] = pd.to_datetime(labeled_df['window'], utc=True)
print(f'Labels v4 shape : {labeled_df.shape}')
print(f'Splits:')
print(labeled_df[SPLIT_COL].value_counts())
print(f'\nDistribución global:')
dist = labeled_df[TARGET_COL].value_counts(normalize=True).sort_index() * 100
labels_map = {0:'Normal', 1:'Baja', 2:'Media', 3:'Alta'}
for k, pct in dist.items():
    print(f'  {labels_map[k]:8s} ({k}): {pct:.1f}%')
print(f'\nPeriodo: {labeled_df["window"].min()} -> {labeled_df["window"].max()}')

baseline_df = pd.read_parquet(IN_BASE)
print(f'\nBaseline v4 shape: {baseline_df.shape}')
print(f'Columnas: {list(baseline_df.columns)}')
print(f'Rutas en baseline: {baseline_df[["source_agent","cloud_provider","region"]].drop_duplicates().shape[0]:,}')

Labels v4 shape : (7628801, 6)
Splits:
split
train    5349594
val      1140847
test     1138360
Name: count, dtype: int64

Distribución global:
  Normal   (0): 86.3%
  Baja     (1): 9.5%
  Media    (2): 2.2%
  Alta     (3): 2.0%

Periodo: 2026-04-30 00:00:00+00:00 -> 2026-09-30 23:30:00+00:00

Baseline v4 shape: (3105, 6)
Columnas: ['source_agent', 'cloud_provider', 'region', 'p95', 'mad', 'metric']
Rutas en baseline: 1,035


In [3]:

PROVIDER_MAP = {
    'aws':   [f for f in ARCHIVOS if '/te_aws_'   in f.lower() or 'te_aws_' in os.path.basename(f).lower()],
    'azure': [f for f in ARCHIVOS if '/te_azure_' in f.lower() or 'te_azure_' in os.path.basename(f).lower()],
    'gcp':   [f for f in ARCHIVOS if '/te_gcp_'   in f.lower() or 'te_gcp_' in os.path.basename(f).lower()],
}
print('Archivos por proveedor:')
for prov, files in PROVIDER_MAP.items():
    print(f'  {prov.upper():5s}: {len(files)} archivos')

net_records = []

for prov, files in PROVIDER_MAP.items():
    print(f'\n[{prov.upper()}] {len(files)} archivos...')
    for fi, f in enumerate(files):
        tbl = pq.read_table(f, columns=['_time','source_agent','cloud_provider','region','metric','value'])
        df  = tbl.to_pandas(); del tbl; gc.collect()

        df['_time'] = pd.to_datetime(df['_time'], utc=True)
        df = df[(df['_time'] >= BD_END) & (df['metric'].isin(METRICS_NET))].copy()
        if df.empty:
            del df; gc.collect()
            continue

        df['window'] = df['_time'].dt.floor('30min')

        grp = df.groupby(['source_agent','cloud_provider','region','metric','window'])['value']
        stats = grp.agg(
            p50    = lambda x: float(np.percentile(x, 50)),
            p95    = lambda x: float(np.percentile(x, 95)),
            mad    = lambda x: float(np.median(np.abs(x - np.median(x)))),
            mean   = 'mean',
            std    = 'std',
            n_samp = 'count',
        ).reset_index()

        net_records.append(stats)
        del df, stats; gc.collect()
        total_rows = sum(len(r) for r in net_records)
        print(f'  [{fi+1}/{len(files)}] {os.path.basename(f)} — acum. {total_rows:,} filas')

    print(f'  ✅ {prov.upper()} listo')

net_df = pd.concat(net_records, ignore_index=True)
del net_records; gc.collect()

print(f'\nnet_df shape: {net_df.shape}')
print(f'Columnas    : {list(net_df.columns)}')

Archivos por proveedor:
  AWS  : 24 archivos
  AZURE: 24 archivos
  GCP  : 24 archivos

[AWS] 24 archivos...
  [1/24] te_aws_raw_apr_w1.parquet — acum. 58,320 filas
  [2/24] te_aws_raw_aug_w1.parquet — acum. 466,556 filas
  [3/24] te_aws_raw_aug_w2.parquet — acum. 874,034 filas
  [4/24] te_aws_raw_aug_w3.parquet — acum. 1,280,816 filas
  [5/24] te_aws_raw_aug_w4.parquet — acum. 1,688,658 filas
  [6/24] te_aws_raw_aug_w5.parquet — acum. 2,096,825 filas
  [7/24] te_aws_raw_jul_w1.parquet — acum. 2,503,369 filas
  [8/24] te_aws_raw_may_w1.parquet — acum. 2,910,637 filas
  [9/24] te_aws_raw_may_w2.parquet — acum. 3,317,414 filas
  [10/24] te_aws_raw_may_w3.parquet — acum. 3,725,357 filas
  [11/24] te_aws_raw_may_w4.parquet — acum. 4,126,181 filas
  [12/24] te_aws_raw_may_w5.parquet — acum. 4,359,459 filas
  [13/24] te_aws_raw_sep_w1.parquet — acum. 4,820,787 filas
  [14/24] te_aws_raw_sep_w2.parquet — acum. 5,227,933 filas
  [15/24] te_aws_raw_sep_w3.parquet — acum. 5,636,121 filas
  [16/2

In [4]:

net_df['metric_short'] = net_df['metric'].map(METRIC_SHORT)

base_lookup = baseline_df[['source_agent','cloud_provider','region','metric','p95','mad']].copy()
base_lookup = base_lookup.rename(columns={'p95': 'base_p95', 'mad': 'base_mad'})

net_df = net_df.merge(base_lookup, on=['source_agent','cloud_provider','region','metric'], how='left')

net_df['delta_p95'] = (net_df['p95'] - net_df['base_p95']) / (net_df['base_p95'] + EPS)
net_df['delta_mad'] = (net_df['mad'] - net_df['base_mad']) / (net_df['base_mad'] + EPS)

value_cols = ['p50','p95','mad','mean','std','n_samp','delta_p95','delta_mad']

wide_df = net_df.pivot_table(
    index=INDEX_COLS, columns='metric_short', values=value_cols, aggfunc='first',
)
wide_df.columns = [f'{stat}_{met}' for stat, met in wide_df.columns]
wide_df = wide_df.reset_index()
wide_df['window'] = pd.to_datetime(wide_df['window'], utc=True)

del net_df, base_lookup; gc.collect()

print(f'wide_df shape: {wide_df.shape}')
print(f'Columnas ({len(wide_df.columns)}):')
for c in sorted(wide_df.columns):
    print(f'  {c}')

wide_df shape: (7628801, 28)
Columnas (28):
  cloud_provider
  delta_mad_jitter
  delta_mad_lat
  delta_mad_loss
  delta_p95_jitter
  delta_p95_lat
  delta_p95_loss
  mad_jitter
  mad_lat
  mad_loss
  mean_jitter
  mean_lat
  mean_loss
  n_samp_jitter
  n_samp_lat
  n_samp_loss
  p50_jitter
  p50_lat
  p50_loss
  p95_jitter
  p95_lat
  p95_loss
  region
  source_agent
  std_jitter
  std_lat
  std_loss
  window


In [5]:

LAG_COLS = [c for c in wide_df.columns
            if c not in INDEX_COLS
            and not c.startswith('n_samp')]

print(f'Columnas a laguear: {len(LAG_COLS)}')
print('  ' + ', '.join(sorted(LAG_COLS)))

lag1_src = wide_df[INDEX_COLS + LAG_COLS].copy()
lag1_src['window'] = lag1_src['window'] + WINDOW
lag1_src = lag1_src.rename(columns={c: f'lag1_{c}' for c in LAG_COLS})

lag2_src = wide_df[INDEX_COLS + LAG_COLS].copy()
lag2_src['window'] = lag2_src['window'] + 2 * WINDOW
lag2_src = lag2_src.rename(columns={c: f'lag2_{c}' for c in LAG_COLS})

lag48_src = wide_df[INDEX_COLS + LAG_COLS].copy()
lag48_src['window'] = lag48_src['window'] + 48 * WINDOW
lag48_src = lag48_src.rename(columns={c: f'lag48_{c}' for c in LAG_COLS})

wide_v6 = (wide_df
           .merge(lag1_src,  on=INDEX_COLS, how='left')
           .merge(lag2_src,  on=INDEX_COLS, how='left')
           .merge(lag48_src, on=INDEX_COLS, how='left'))

del lag1_src, lag2_src, lag48_src; gc.collect()

n_total = len(wide_v6)
for lag_prefix, label in [('lag1_', 't-1'), ('lag2_', 't-2'), ('lag48_', 't-48')]:
    lag_c = [c for c in wide_v6.columns if c.startswith(lag_prefix)]
    n_nan = wide_v6[lag_c].isnull().any(axis=1).sum()
    print(f'NaN lag {label:4s}  : {n_nan:,} / {n_total:,} ({100*n_nan/n_total:.1f}%)')

print(f'\nwide_v6 shape : {wide_v6.shape}')
print(f'Sin lag336_*  : {sum(1 for c in wide_v6.columns if c.startswith("lag336_"))} (esperado: 0)')
print(f'W_v6={W_v6}min >= H={H}min  ✅')

Columnas a laguear: 21
  delta_mad_jitter, delta_mad_lat, delta_mad_loss, delta_p95_jitter, delta_p95_lat, delta_p95_loss, mad_jitter, mad_lat, mad_loss, mean_jitter, mean_lat, mean_loss, p50_jitter, p50_lat, p50_loss, p95_jitter, p95_lat, p95_loss, std_jitter, std_lat, std_loss
NaN lag t-1   : 3,915 / 7,628,801 (0.1%)
NaN lag t-2   : 5,887 / 7,628,801 (0.1%)
NaN lag t-48  : 65,640 / 7,628,801 (0.9%)

wide_v6 shape : (7628801, 91)
Sin lag336_*  : 0 (esperado: 0)
W_v6=90min >= H=60min  ✅


In [6]:

wide_v6['hour']     = wide_v6['window'].dt.hour
wide_v6['hour_sin'] = np.sin(2 * np.pi * wide_v6['hour'] / 24)
wide_v6['hour_cos'] = np.cos(2 * np.pi * wide_v6['hour'] / 24)
wide_v6['dow']      = wide_v6['window'].dt.dayofweek
wide_v6['dow_sin']  = np.sin(2 * np.pi * wide_v6['dow'] / 7)
wide_v6['dow_cos']  = np.cos(2 * np.pi * wide_v6['dow'] / 7)
wide_v6['is_weekend'] = (wide_v6['dow'] >= 5).astype(np.int8)
wide_v6.drop(columns=['hour', 'dow'], inplace=True)

TEMPORAL_COLS = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']
print(f'Temporal features: {TEMPORAL_COLS}')
print(f'wide_v6 shape    : {wide_v6.shape}')
vc = wide_v6['is_weekend'].value_counts().sort_index()
for v, cnt in vc.items():
    label = 'fin_semana' if v == 1 else 'entre_semana'
    print(f'  {v} ({label}): {cnt:,}  ({100*cnt/len(wide_v6):.1f}%)')

Temporal features: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']
wide_v6 shape    : (7628801, 96)
  0 (entre_semana): 5,446,986  (71.4%)
  1 (fin_semana): 2,181,815  (28.6%)


In [7]:

bgp_records = []

for prov, files in PROVIDER_MAP.items():
    print(f'[{prov.upper()}] BGP...')
    for f in files:
        try:
            tbl = pq.read_table(f, columns=['_time','cloud_provider','region','metric','value'])
        except Exception as e:
            print(f'  ⚠️  {os.path.basename(f)}: {e}')
            continue

        df = tbl.to_pandas(); del tbl; gc.collect()
        df['_time'] = pd.to_datetime(df['_time'], utc=True)
        df = df[(df['_time'] >= BD_END) & (df['metric'].isin(METRICS_BGP))].copy()
        if df.empty:
            del df; gc.collect()
            continue

        df['window'] = df['_time'].dt.floor('30min')
        grp      = df.groupby(['cloud_provider','region','metric','window'])['value']
        agg_sum  = grp.sum().rename('val_sum')
        agg_mean = grp.mean().rename('val_mean')
        agg_df   = pd.concat([agg_sum, agg_mean], axis=1).reset_index()
        bgp_records.append(agg_df)
        del df, agg_df; gc.collect()

if bgp_records:
    bgp_df = pd.concat(bgp_records, ignore_index=True)
    del bgp_records; gc.collect()

    bgp_df['bgp_val'] = bgp_df.apply(
        lambda r: r['val_mean'] if 'reachability' in r['metric'] else r['val_sum'], axis=1)
    bgp_df['metric_short'] = bgp_df['metric'].map(BGP_METRIC_SHORT)

    bgp_wide = bgp_df.pivot_table(
        index=['cloud_provider','region','window'],
        columns='metric_short', values='bgp_val', aggfunc='first',
    ).reset_index()
    bgp_wide['window'] = pd.to_datetime(bgp_wide['window'], utc=True)

    print(f'BGP features shape: {bgp_wide.shape}')
    print(f'Columnas BGP: {list(bgp_wide.columns)}')
else:
    print('⚠️  Sin datos BGP — se omite esta feature')
    bgp_wide = None

[AWS] BGP...
[AZURE] BGP...
[GCP] BGP...
BGP features shape: (162945, 6)
Columnas BGP: ['cloud_provider', 'region', 'window', 'bgp_path_changes', 'bgp_reachability', 'bgp_updates']


In [8]:

feat_df = wide_v6.merge(
    labeled_df[['source_agent','cloud_provider','region','window','label_ventana','split']],
    on=['source_agent','cloud_provider','region','window'],
    how='inner'
)

if bgp_wide is not None:
    feat_df = feat_df.merge(bgp_wide, on=['cloud_provider','region','window'], how='left')

print(f'Feature dataset shape (post-join): {feat_df.shape}')
print(f'\nCobertura vs labels v4:')
for split in ['train', 'val', 'test']:
    n_label = (labeled_df[SPLIT_COL] == split).sum()
    n_feat  = (feat_df[SPLIT_COL] == split).sum()
    pct = 100 * n_feat / n_label if n_label > 0 else 0
    print(f'  {split:5s}: labels={n_label:>10,}  features={n_feat:>10,}  ({pct:.1f}%)')

print('\nOrdenando por ruta + ventana (requerido para shifts)...')
feat_df = feat_df.sort_values(ROUTE_KEYS + ['window']).reset_index(drop=True)

grp = feat_df.groupby(ROUTE_KEYS)[TARGET_COL]
feat_df['y_lag1'] = feat_df[TARGET_COL].astype(float)
feat_df['y_lag2'] = grp.shift(1).astype(float)
feat_df['y_lag3'] = grp.shift(2).astype(float)
gc.collect()
print(f'y_lag1/2/3 computados. y_lag1 NaN: {feat_df["y_lag1"].isna().sum():,} (esperado: 0)')
assert (feat_df['y_lag1'] == feat_df[TARGET_COL].astype(float)).all(), 'ERROR: y_lag1 != label_ventana'
print('  Anti-leakage: y_lag1 == label_ventana ✅')

prev_label = feat_df.groupby(ROUTE_KEYS)[TARGET_COL].shift(1)
feat_df['__not_same__']  = (feat_df[TARGET_COL] != prev_label).astype(int)
del prev_label; gc.collect()
feat_df['__cumchange__'] = feat_df['__not_same__'].cumsum()
feat_df['streak_length'] = (
    feat_df.groupby(ROUTE_KEYS + ['__cumchange__'])
    .cumcount().add(1).astype(np.int32)
)
feat_df.drop(columns=['__not_same__', '__cumchange__'], inplace=True)
gc.collect()
print(f'streak_length: max={feat_df["streak_length"].max()}  median={feat_df["streak_length"].median():.1f}')

feat_df['d_latency'] = feat_df['mean_lat']    - feat_df['lag1_mean_lat']
feat_df['d_jitter']  = feat_df['mean_jitter'] - feat_df['lag1_mean_jitter']
feat_df['d_loss']    = feat_df['mean_loss']   - feat_df['lag1_mean_loss']
print(f'd_latency/jitter/loss computados.')

feat_df['latency_ratio_lag48'] = feat_df['mean_lat'] / (feat_df['lag48_mean_lat'].abs() + EPS)
print(f'latency_ratio_lag48: mediana={feat_df["latency_ratio_lag48"].dropna().median():.4f} (esperado ~1.0)')

print(f'\nfeat_df shape final: {feat_df.shape}')

Feature dataset shape (post-join): (7628801, 101)

Cobertura vs labels v4:
  train: labels= 5,349,594  features= 5,349,594  (100.0%)
  val  : labels= 1,140,847  features= 1,140,847  (100.0%)
  test : labels= 1,138,360  features= 1,138,360  (100.0%)

Ordenando por ruta + ventana (requerido para shifts)...
y_lag1/2/3 computados. y_lag1 NaN: 0 (esperado: 0)
  Anti-leakage: y_lag1 == label_ventana ✅
streak_length: max=5197  median=39.0
d_latency/jitter/loss computados.
latency_ratio_lag48: mediana=0.9999 (esperado ~1.0)

feat_df shape final: (7628801, 109)


In [9]:

META_COLS   = INDEX_COLS + [TARGET_COL, SPLIT_COL]
N_SAMP_COLS = [c for c in feat_df.columns if c.startswith('n_samp')]
feat_cols_v6 = [c for c in feat_df.columns
                if c not in META_COLS
                and not c.startswith('n_samp')]

print('=== Feature Engineering v6 — Resumen ===')
print(f'Dataset shape    : {feat_df.shape}')
print(f'Total columnas   : {len(feat_df.columns)}')
print(f'  Meta           : {len(META_COLS)}  {META_COLS}')
print(f'  n_samp cols    : {len(N_SAMP_COLS)}')
print(f'  FEATURES MODELO: {len(feat_cols_v6)}')

n_lag336 = sum(1 for c in feat_df.columns if c.startswith('lag336_'))
assert n_lag336 == 0, f'ERROR: {n_lag336} columnas lag336_* presentes'
print(f'\n[OK] lag336_* eliminados: {n_lag336}  (D12-REV)')

v5_feats = ['y_lag1','y_lag2','y_lag3','streak_length','d_latency','d_jitter','d_loss','latency_ratio_lag48']
print(f'\n[OK] Features v5 ({len(v5_feats)}):')
for f in v5_feats:
    assert f in feat_df.columns, f'ERROR: falta {f}'
    n_nan = feat_df[f].isna().sum()
    print(f'  {f:25s}: NaN={n_nan:>8,}  ({100*n_nan/len(feat_df):5.2f}%)')

print(f'\n=== Splits ===')
for split in ['train', 'val', 'test']:
    n = (feat_df[SPLIT_COL] == split).sum()
    print(f'  {split:5s}: {n:>10,}')

print(f'\n=== Distribución labels ===')
labels_map = {0:'Normal', 1:'Baja', 2:'Media', 3:'Alta'}
dist = feat_df[TARGET_COL].value_counts(normalize=True).sort_index() * 100
for k, pct in dist.items():
    print(f'  {labels_map[k]:6s} ({k}): {pct:.1f}%')

print(f'\n=== Composición features ({len(feat_cols_v6)}) ===')
n_lag1  = sum(1 for c in feat_cols_v6 if c.startswith('lag1_'))
n_lag2  = sum(1 for c in feat_cols_v6 if c.startswith('lag2_'))
n_lag48 = sum(1 for c in feat_cols_v6 if c.startswith('lag48_'))
n_y_lag = sum(1 for c in feat_cols_v6 if c.startswith('y_lag'))
n_temp  = sum(1 for c in feat_cols_v6 if c in ['hour_sin','hour_cos','dow_sin','dow_cos','is_weekend'])
n_bgp   = sum(1 for c in feat_cols_v6 if c.startswith('bgp_'))
n_dmet  = sum(1 for c in feat_cols_v6 if c.startswith('d_'))
n_ratio = sum(1 for c in feat_cols_v6 if c == 'latency_ratio_lag48')
n_stk   = sum(1 for c in feat_cols_v6 if c == 'streak_length')
n_base  = len(feat_cols_v6) - n_lag1 - n_lag2 - n_lag48 - n_y_lag - n_temp - n_bgp - n_dmet - n_ratio - n_stk
print(f'  Base de red (p50/p95/mad/mean/std/delta) : {n_base}')
print(f'  Lag t-1                                  : {n_lag1}')
print(f'  Lag t-2                                  : {n_lag2}')
print(f'  Lag t-48 (24h)                           : {n_lag48}')
print(f'  y_lag (autoregresión etiquetas)  [v5]    : {n_y_lag}')
print(f'  streak_length                    [v5]    : {n_stk}')
print(f'  d_metric (primera diferencia)    [v5]    : {n_dmet}')
print(f'  latency_ratio_lag48              [v5]    : {n_ratio}')
print(f'  HOD+DOW temporal                         : {n_temp}')
print(f'  BGP                                      : {n_bgp}')
print(f'  ────────────────────────────────────────────')
print(f'  TOTAL MODELO                             : {len(feat_cols_v6)}')

print(f'\n=== Nulos en features v5 ===')
nulls = feat_df[v5_feats].isnull().sum()
nulls = nulls[nulls > 0]
if not nulls.empty:
    print(nulls.sort_values(ascending=False).to_string())
else:
    print('  Sin nulos adicionales (solo NaN esperados en lag features de primeras ventanas)')

=== Feature Engineering v6 — Resumen ===
Dataset shape    : (7628801, 109)
Total columnas   : 109
  Meta           : 6  ['source_agent', 'cloud_provider', 'region', 'window', 'label_ventana', 'split']
  n_samp cols    : 3
  FEATURES MODELO: 100

[OK] lag336_* eliminados: 0  (D12-REV)

[OK] Features v5 (8):
  y_lag1                   : NaN=       0  ( 0.00%)
  y_lag2                   : NaN=   1,035  ( 0.01%)
  y_lag3                   : NaN=   2,070  ( 0.03%)
  streak_length            : NaN=       0  ( 0.00%)
  d_latency                : NaN=   3,725  ( 0.05%)
  d_jitter                 : NaN=   3,741  ( 0.05%)
  d_loss                   : NaN=   3,174  ( 0.04%)
  latency_ratio_lag48      : NaN=  65,481  ( 0.86%)

=== Splits ===
  train:  5,349,594
  val  :  1,140,847
  test :  1,138,360

=== Distribución labels ===
  Normal (0): 86.3%
  Baja   (1): 9.5%
  Media  (2): 2.2%
  Alta   (3): 2.0%

=== Composición features (100) ===
  Base de red (p50/p95/mad/mean/std/delta) : 21
  Lag t-1 

In [10]:

META_COLS    = INDEX_COLS + [TARGET_COL, SPLIT_COL]
N_SAMP_COLS  = [c for c in feat_df.columns if c.startswith('n_samp')]
feat_cols_v6 = [c for c in feat_df.columns
                if c not in META_COLS and not c.startswith('n_samp')]

splits = feat_df[SPLIT_COL].value_counts().to_dict()

print('Guardando te_features_v6.parquet...')
feat_df.to_parquet(OUT_FEAT, index=False)
print(f'  Guardado : {OUT_FEAT}')
print(f'  Shape    : {feat_df.shape}')
print(f'  Features : {len(feat_cols_v6)}  (para modelo)')
print(f'  Splits   : {splits}')

if MLFLOW_OK:
    v5_feats = ['y_lag1','y_lag2','y_lag3','streak_length',
                'd_latency','d_jitter','d_loss','latency_ratio_lag48']
    n_test_labels = (labeled_df[SPLIT_COL] == 'test').sum()
    cob_test = round(100 * splits.get('test', 0) / n_test_labels, 2) if n_test_labels > 0 else 0

    with mlflow.start_run(run_name='feature_engineering_v6'):
        mlflow.log_param('notebook',       '04 feature_engineering v6.ipynb')
        mlflow.log_param('oe',             'OE2')
        mlflow.log_param('input_labels',   'te_labeled_ventanas_v4.parquet')
        mlflow.log_param('input_baseline', 'te_baseline_stats_v4.parquet')
        mlflow.log_param('n_archivos_raw', len(ARCHIVOS))
        mlflow.log_param('bd_end',         str(BD_END.date()))
        mlflow.log_param('lag336_removed', True)
        mlflow.log_param('lags',           't-1,t-2,t-48')

        mlflow.log_metric('n_ventanas_total',  len(feat_df))
        mlflow.log_metric('n_features_modelo', len(feat_cols_v6))
        mlflow.log_metric('n_train',           splits.get('train', 0))
        mlflow.log_metric('n_val',             splits.get('val',   0))
        mlflow.log_metric('n_test',            splits.get('test',  0))
        mlflow.log_metric('cobertura_test_pct', cob_test)

        for feat in v5_feats:
            if feat in feat_df.columns:
                mlflow.log_metric(f'nan_pct_{feat}',
                                  round(100 * feat_df[feat].isna().mean(), 4))

        mlflow.set_tag('output_file',  OUT_FEAT)
        mlflow.set_tag('feature_cols', ','.join(sorted(feat_cols_v6)))

    print(f'\n✅ MLflow run registrado: feature_engineering_v6')
    print(f'   URI        : http://localhost:6000')
    print(f'   Experimento: latam2cloud')
else:
    print('\n⚠️  MLflow no disponible — run no registrado (resultado en parquet)')

print(f'\n=== Cambios v5 → v6 (resumen) ===')
print(f'  INPUT NUEVO : te_labeled_ventanas_v4.parquet (162d vs 83d en v3/v4/v5)')
print(f'  BASELINE    : te_baseline_stats_v4.parquet   (Apr 23-29, BD=7d)')
print(f'  ARCHIVOS    : 72 parquets (Apr-Sep vs 39 May-Jul)')
print(f'  BD_END      : 2026-04-30 (vs 2026-05-08 en v4)')
print(f'  FEATURES    : mismo set que v5 (~100) — NO nuevos features')
print(f'  lag336_*    : eliminados (D12-REV, Strobl et al. 2007, BMC Bioinf. 8:25)')
print(f'\nSiguiente: 05 modelado v10.ipynb  (OE2 · Plan de Tesis)')
print(f'  Punto de partida: Ensemble D38 (RF v5.1×0.80 + LGB v7×0.20, LOG-25)')
print(f'  Criterio H2 (anti-HARKing, Kerr 1998, PSPR 3(3):196):')
print(f'    QWK >= 0.65  AND  RecAlta >= 0.55  (simultáneamente, en val primero)')

Guardando te_features_v6.parquet...
  Guardado : /srv/work/te_features_v6.parquet
  Shape    : (7628801, 109)
  Features : 100  (para modelo)
  Splits   : {'train': 5349594, 'val': 1140847, 'test': 1138360}
🏃 View run feature_engineering_v6 at: http://localhost:6000/#/experiments/681062908553326840/runs/3bf4954117ef42189a648a869414cff1
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840

✅ MLflow run registrado: feature_engineering_v6
   URI        : http://localhost:6000
   Experimento: latam2cloud

=== Cambios v5 → v6 (resumen) ===
  INPUT NUEVO : te_labeled_ventanas_v4.parquet (162d vs 83d en v3/v4/v5)
  BASELINE    : te_baseline_stats_v4.parquet   (Apr 23-29, BD=7d)
  ARCHIVOS    : 72 parquets (Apr-Sep vs 39 May-Jul)
  BD_END      : 2026-04-30 (vs 2026-05-08 en v4)
  FEATURES    : mismo set que v5 (~100) — NO nuevos features
  lag336_*    : eliminados (D12-REV, Strobl et al. 2007, BMC Bioinf. 8:25)

Siguiente: 05 modelado v10.ipynb  (OE2 · Plan de Tesis)
  